In [8]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

packages = ['opencv-python', 'tensorflow', 'keras', 'numpy', 'matplotlib']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        install(package)

Installing opencv-python...
tensorflow already installed
keras already installed
numpy already installed
matplotlib already installed


In [9]:
from __future__ import print_function
import os
import shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
import math
import random
import time
from collections import defaultdict

import keras
from keras.models import Model
from keras.layers import Dense, Input, Conv3D, MaxPooling3D, concatenate, Flatten
from keras.optimizers import SGD
import tensorflow as tf

print(f'Python version: {os.sys.version}')
print(f'OpenCV version: {cv2.__version__}')
print(f'Keras version: {keras.__version__}')
print(f'TensorFlow version: {tf.__version__}')

Python version: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
OpenCV version: 4.12.0
Keras version: 3.14.0
TensorFlow version: 2.21.0


In [10]:
def progress_bar(count, total, title, completed=0):
    terminal_size = 80
    percentage = int(100.0 * count / total)
    length_bar = max([4, terminal_size - len(title) - len(str(total)) - len(str(count)) - len(str(percentage)) - 10])
    filled_len = int(length_bar * count / total)
    bar = '█' * filled_len + ' ' * (length_bar - filled_len)
    print(f'{title} [{bar}] {percentage} % ({count}/{total})', end='\r')
    if completed:
        print(f'{title} [{bar}] {percentage} % ({count}/{total})')

def make_new_path(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def make_path(path):
    os.makedirs(path, exist_ok=True)

def draw_hsv(flow, normalization=30, inv=True):
    fy = flow[:,:,1]
    fx = flow[:,:,0]
    h, w = fx.shape
    ang = np.arctan2(fy, fx) + np.pi
    v = normalization * np.sqrt(fx*fx + fy*fy) / math.sqrt(2)
    hsv = np.zeros((h, w, 3), np.uint8)
    hsv[...,0] = ang*(180/np.pi/2)
    hsv[...,1] = 255
    hsv[...,2] = np.minimum(v*4, 255)
    visualization = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    if inv:
        visualization = 255*np.ones(visualization.shape, dtype=np.uint8) - visualization
    return visualization.astype('uint8')

print("Utility functions loaded successfully!")

Utility functions loaded successfully!


In [11]:
def extract_rgb_from_video(video_path, output_path, height=120):
    """
    Extract RGB frames from a video file
    """
    make_new_path(output_path)
    
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        h = frame.shape[0]
        w = frame.shape[1]
        new_w = int(w * height / h)
        frame_resized = cv2.resize(frame, (new_w, height))
        
        frame_path = os.path.join(output_path, f'{frame_count:08d}.png')
        cv2.imwrite(frame_path, frame_resized)
        frame_count += 1
    
    cap.release()
    print(f"Extracted {frame_count} frames from video")
    return frame_count

def compute_optical_flow(rgb_frames_path, flow_output_path, size_data=[100, 100]):
    """
    Compute optical flow using OpenCV
    """
    make_new_path(flow_output_path)
    
    frames = sorted([f for f in os.listdir(rgb_frames_path) if f.endswith('.png')])
    max_flow_values = []
    
    if len(frames) < 2:
        print("Not enough frames to compute optical flow")
        return None
    
    old_frame = cv2.imread(os.path.join(rgb_frames_path, frames[0]), 0)
    
    for i in range(1, len(frames)):
        progress_bar(i, len(frames), 'Computing Optical Flow')
        new_frame = cv2.imread(os.path.join(rgb_frames_path, frames[i]), 0)
        
        flow = cv2.calcOpticalFlowFarneback(old_frame, new_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        flow_path = os.path.join(flow_output_path, f'{i:08d}')
        np.save(flow_path, flow)
        
        max_flow_values.append(np.abs(flow).max())
        old_frame = new_frame.copy()
    
    progress_bar(len(frames), len(frames), 'Computing Optical Flow', completed=1)
    return max_flow_values

def get_flow_normalization(flow_values, method='99.99%'):
    """
    Calculate optical flow normalization factor
    """
    max_flow = np.max(flow_values)
    mean_flow = np.mean(flow_values)
    std_flow = np.std(flow_values)
    
    if method == 'Max':
        norm_flow = max_flow
    elif method == '99.99%':
        norm_flow = sorted(flow_values)[int(0.9999*len(flow_values))]
    else:
        norm_flow = mean_flow + 3*std_flow
    
    print(f'Flow: Max {max_flow:.2f} - Mean {mean_flow:.2f} +- {std_flow:.2f}')
    print(f'Normalization method: {method}, flow divided by {norm_flow:.2f}')
    return norm_flow

print("Video processing functions loaded successfully!")

Video processing functions loaded successfully!


In [12]:
def make_branch_model(temporal_dim, width, height, channels, nb_class):
    """
    Build a single branch of the Siamese model
    """
    input_data = Input(shape=(temporal_dim, width, height, channels))
    
    output = Conv3D(30, kernel_size=(3,3,3), padding='same', activation='relu')(input_data)
    output = MaxPooling3D(pool_size=(2,2,2))(output)
    
    output = Conv3D(60, kernel_size=(3,3,3), padding='same', activation='relu')(output)
    output = MaxPooling3D(pool_size=(2,2,2))(output)
    
    output = Conv3D(80, kernel_size=(3,3,3), padding='same', activation='relu')(output)
    output = MaxPooling3D(pool_size=(2,2,2))(output)
    
    output = Flatten()(output)
    output = Dense(500, activation='relu')(output)
    
    return output, input_data

def make_siamese_model(temporal_dim, width, height, nb_class):
    """
    Build the Siamese CNN model with two branches (RGB and Optical Flow)
    """
    rgb_model, rgb_input = make_branch_model(temporal_dim, width, height, 3, nb_class)
    flow_model, flow_input = make_branch_model(temporal_dim, width, height, 2, nb_class)
    
    merged = concatenate([rgb_model, flow_model])
    
    output = Dense(nb_class, activation='softmax')(merged)
    
    model = Model(inputs=[rgb_input, flow_input], outputs=output)
    
    sgd = SGD(learning_rate=0.001, weight_decay=1e-6, momentum=0.5, nesterov=True)
    model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
    
    return model

print("Model architecture functions loaded successfully!")

Model architecture functions loaded successfully!


In [13]:
TEMPORAL_DIM = 100
WIDTH = 100
HEIGHT = 100
SIZE_DATA = [100, 100]

CLASS_INDEX = [
    'ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam',
    'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress'
]

print(f"Total classes: {len(CLASS_INDEX)}")
print(f"Configuration: Temporal={TEMPORAL_DIM}, Width={WIDTH}, Height={HEIGHT}")

Total classes: 10
Configuration: Temporal=100, Width=100, Height=100


In [14]:
print("Creating Siamese CNN model...")
model = make_siamese_model(TEMPORAL_DIM, WIDTH, HEIGHT, len(CLASS_INDEX))
model.summary()

Creating Siamese CNN model...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 100, 2)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_6 (Conv3D)   │ (None, 100, 100,  │      2,460 │ input_layer_2[0]… │
│                     │ 100, 30)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_9 (Conv3D)   │ (None, 100, 100,  │      1,650 │ input_layer_3[0]… │
│                     │ 100, 30)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_6     │ (None, 50, 50,    │          0 │ conv3d_6[0][0]    │
│ (MaxPooling3D)      │ 50, 30)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_9     │ (None, 50, 50,    │          0 │ conv3d_9[0][0]    │
│ (MaxPooling3D)      │ 50, 30)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_7 (Conv3D)   │ (None, 50, 50,    │     48,660 │ max_pooling3d_6[… │
│                     │ 50, 60)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_10 (Conv3D)  │ (None, 50, 50,    │     48,660 │ max_pooling3d_9[… │
│                     │ 50, 60)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_7     │ (None, 25, 25,    │          0 │ conv3d_7[0][0]    │
│ (MaxPooling3D)      │ 25, 60)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_10    │ (None, 25, 25,    │          0 │ conv3d_10[0][0]   │
│ (MaxPooling3D)      │ 25, 60)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_8 (Conv3D)   │ (None, 25, 25,    │    129,680 │ max_pooling3d_7[… │
│                     │ 25, 80)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_11 (Conv3D)  │ (None, 25, 25,    │    129,680 │ max_pooling3d_10… │
│                     │ 25, 80)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_8     │ (None, 12, 12,    │          0 │ conv3d_8[0][0]    │
│ (MaxPooling3D)      │ 12, 80)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_11    │ (None, 12, 12,    │          0 │ conv3d_11[0][0]   │
│ (MaxPooling3D)      │ 12, 80)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 138240)    │          0 │ max_pooling3d_8[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 138240)    │          0 │ max_pooling3d_11… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 500)       │ 69,120,500 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 500)       │ 69,120,500 │ flatten_3[0][0] 

 Total params: 138,611,800 (528.76 MB)

 Trainable params: 138,611,800 (528.76 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
weights_file = 'Siamese_model.hdf5'

if os.path.exists(weights_file):
    print(f"Loading pre-trained weights from {weights_file}...")
    model.load_weights(weights_file)
    print("Weights loaded successfully!")
else:
    print(f"Warning: Weights file '{weights_file}' not found.")
    print("For video classification, please provide trained weights.")

For video classification, please provide trained weights.


In [16]:
print("""
╔══════════════════════════════════════════════════════════════╗
║    VIDEO ACTION RECOGNITION - CLASSIFICATION NOTEBOOK        ║
╚══════════════════════════════════════════════════════════════╝

QUICK START GUIDE:
1. Upload or prepare your video file
2. Update video_path variable below
3. Run the classification cells
4. View predictions and confidence scores

SUPPORTED VIDEO FORMATS:
- MP4, AVI, MOV, FLV, WMV, MKV, and more

MODEL FEATURES:
✓ Siamese CNN with RGB + Optical Flow branches
✓ Trained on UCF101 dataset (101 action classes)
✓ Temporal dimension processing (100 frames)
✓ Spatial segmentation with ROI extraction
✓ Real-time classification with confidence scores

NOTE: Pre-trained weights are required for accurate results.
Download from the repository or train your own model.
""")


╔══════════════════════════════════════════════════════════════╗
║    VIDEO ACTION RECOGNITION - CLASSIFICATION NOTEBOOK        ║
╚══════════════════════════════════════════════════════════════╝

QUICK START GUIDE:
1. Upload or prepare your video file
2. Update video_path variable below
3. Run the classification cells
4. View predictions and confidence scores

SUPPORTED VIDEO FORMATS:
- MP4, AVI, MOV, FLV, WMV, MKV, and more

MODEL FEATURES:
✓ Siamese CNN with RGB + Optical Flow branches
✓ Trained on UCF101 dataset (101 action classes)
✓ Temporal dimension processing (100 frames)
✓ Spatial segmentation with ROI extraction
✓ Real-time classification with confidence scores

NOTE: Pre-trained weights are required for accurate results.
Download from the repository or train your own model.

